# Séance 9 — Canevas du projet et préparation de la soutenance

Ce notebook n'est pas un atelier : c'est le **squelette de votre rendu**. Remplacez chaque
section par votre travail. La structure suit l'ordre de la soutenance, ce qui vous évite
de refaire le plan au dernier moment.

---

## Identification

| | |
|---|---|
| Groupe | *numéro et noms* |
| Sujet | *domaine attribué, voir la fiche Sujets de projet* |
| Question traitée | *reprise de la fiche, ou reformulée si vous l'affinez* |
| Date de soutenance | *jour 5, après-midi* |

> **Format.** 15 minutes de présentation, 10 minutes de questions devant un jury unique.
> Les questions sont posées individuellement : chaque membre du groupe doit pouvoir
> répondre sur l'intégralité du travail, y compris les parties qu'il n'a pas présentées.
> **Dépôt du rendu : jour 5, 13 h 30.**

> **Convention de nommage du module.** Le code est écrit en anglais et suit la PEP 8 :
> fonctions et variables en `snake_case`, constantes en `MAJUSCULES`. Les **noms de colonnes**
> restent en français parce qu'ils viennent de la source : renommer les colonnes d'un fichier
> d'entrée est une transformation comme une autre, elle se décide et se documente, elle ne se
> fait pas par réflexe. Vous rencontrerez cette situation partout en entreprise.

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from pathlib import Path

# CONFIGURATION GENERIQUE
# Ce notebook doit fonctionner sur Windows, macOS et Linux.
#
# Organisation conseillée :
# mon_projet/
# ├── data/
# │   ├── RAW/              <- placez ici les fichiers CSV d'origine
# │   └── processed/        <- fichiers produits par le notebook
# └── notebooks/
#     └── 09_projet_cantine_TP_generique.ipynb
#
# Vous pouvez aussi placer le notebook directement à la racine du projet.
# Si vos données sont ailleurs, indiquez leur dossier dans DATA_DIR ci-dessous.

def find_project_root(start: Path) -> Path:
    """Remonte les dossiers pour trouver un dossier data/ ou RAW/ connu."""
    for folder in [start, *start.parents]:
        if (folder / "data").is_dir() or (folder / "RAW").is_dir() or (folder / "raw").is_dir():
            return folder
    # Cas courant : le notebook se trouve dans un sous-dossier notebooks/
    if start.name.lower() == "notebooks":
        return start.parent
    return start

ROOT = find_project_root(Path.cwd())

# Détection automatique des emplacements courants.
DATA_CANDIDATES = [
    ROOT / "data" / "RAW",
    ROOT / "data" / "raw",
    ROOT / "data" / "Raw",
    ROOT / "RAW",
    ROOT / "raw",
]

RAW_DIR = next((folder for folder in DATA_CANDIDATES if folder.is_dir()), None)

# SI NÉCESSAIRE : remplacez None par le chemin de votre dossier de CSV.
# Exemples Windows : Path(r"C:\Users\VotreNom\Documents\mon_projet\data\RAW")
# Exemple macOS/Linux : Path.home() / "Documents" / "mon_projet" / "data" / "RAW"
DATA_DIR = None
if DATA_DIR is not None:
    RAW_DIR = Path(DATA_DIR).expanduser()

if RAW_DIR is None:
    # Valeur par défaut à créer si aucun dossier n'a été détecté.
    RAW_DIR = ROOT / "data" / "RAW"

PROCESSED_DIR = ROOT / "data" / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 160)

print("Dossier du projet :", ROOT.resolve())
print("Dossier des CSV :", RAW_DIR.resolve())
print("Dossier de données trouvé :", RAW_DIR.is_dir())

if RAW_DIR.is_dir():
    print("Fichiers présents :")
    for path in sorted(RAW_DIR.iterdir()):
        if path.is_file():
            print(" -", path.name)
else:
    print(
        "Dossier de données introuvable. Créez le dossier data/RAW "
        "dans le projet, placez-y les CSV, ou renseignez DATA_DIR."
    )

def save_dataset(df, name):
    """Enregistre un jeu de données en Parquet, ou en CSV si Parquet n'est pas disponible."""
    try:
        path = PROCESSED_DIR / f"{name}.parquet"
        df.to_parquet(path, index=False)
    except (ImportError, ModuleNotFoundError):
        path = PROCESSED_DIR / f"{name}.csv"
        df.to_csv(path, index=False)
        print("Dépendance Parquet absente : enregistrement en CSV.")
    print("Fichier enregistré :", path, "| dimensions :", df.shape)
    return path

def load_dataset(name):
    """Recharge un jeu de données enregistré précédemment."""
    parquet_path = PROCESSED_DIR / f"{name}.parquet"
    csv_path = PROCESSED_DIR / f"{name}.csv"
    if parquet_path.exists():
        return pd.read_parquet(parquet_path)
    if csv_path.exists():
        return pd.read_csv(csv_path)
    raise FileNotFoundError(
        f"Jeu de données « {name} » introuvable dans {PROCESSED_DIR}. "
        "Exécutez d'abord la cellule qui le crée."
    )

def dataset_exists(name):
    """Indique si un jeu de données existe au format Parquet ou CSV."""
    return (
        (PROCESSED_DIR / f"{name}.parquet").exists()
        or (PROCESSED_DIR / f"{name}.csv").exists()
    )


> **Votre jeu de données vous est fourni.** Chaque groupe reçoit un domaine différent,
> avec ses propres anomalies : les chiffres, les pièges et les décisions à justifier ne
> sont pas les mêmes d'un groupe à l'autre. Comparer vos *méthodes* entre groupes est
> encouragé ; comparer vos *résultats* ne vous mènera nulle part.

> **Aucun sujet ne dit quelles anomalies son jeu contient.** Les trouver est la première
> partie du travail, et la première chose que le jury vérifiera. Commencez par appliquer
> votre `profile_dataframe` du jour 1 : elle a été écrite pour cela.

## 1. Contexte (2 min de présentation)

Le jeu de données fourni pour le groupe Cantine contient quatre tables : les convives, les établissements, les menus et les repas servis. Les repas couvrent la période du **1er janvier 2023 au 30 décembre 2024** pour les dates renseignées.

L’objectif de ce TP est de préparer un jeu de données fiable pour analyser les repas servis, en reliant les repas aux menus, aux convives et aux établissements. La source exacte des fichiers et une question métier officielle plus précise ne sont pas indiquées dans les fichiers fournis : cette présentation décrit donc le périmètre observable, sans inventer d’origine.

Question de travail proposée : **comment obtenir des données suffisamment fiables pour suivre les repas servis par date, établissement, menu, régime et statut ?**


In [7]:

print("ROOT =", ROOT)
print("ROOT existe :", ROOT.exists())
print("Dossier raw :", ROOT / "raw")
print("Dossier raw existe :", (ROOT / "raw").exists())
print("Contenu de ROOT :")

for path in ROOT.iterdir():
    print(path.name)


ROOT = /Users/benjaminrenard/Downloads/Master Data : IA Python Pandas & Numpy/Master-Data-IA
ROOT existe : True
Dossier raw : /Users/benjaminrenard/Downloads/Master Data : IA Python Pandas & Numpy/Master-Data-IA/raw
Dossier raw existe : False
Contenu de ROOT :
02_pandas_exploration.ipynb
.DS_Store
06_agregation_jointures.ipynb
01_environnement_numpy.ipynb
07_pyspark.ipynb
.venv-pyspark
09_projet_canevas.ipynb
04_transformation_encodage.ipynb
03_nettoyage_qualite.ipynb
README.md
.gitignore
05_variables_temporelles.ipynb
.git
data
src


In [ ]:
# Chargement des données brutes : aucun nettoyage à ce stade
import pandas as pd

GROUP_DIR = RAW_DIR
print("Dossier :", GROUP_DIR)
print("Existe :", GROUP_DIR.exists())

print("\nFichiers CSV disponibles :")
for path in sorted(GROUP_DIR.glob("*.csv")):
    size_kb = path.stat().st_size / 1024
    print(f"  {path.name:<28} {size_kb:>8.0f} Ko")

# Charger uniquement les 4 fichiers du groupe Cantine
convives = pd.read_csv(GROUP_DIR / "convives.csv")
etablissements = pd.read_csv(GROUP_DIR / "etablissements.csv")
menus = pd.read_csv(GROUP_DIR / "menus.csv")
repas_servis = pd.read_csv(GROUP_DIR / "repas_servis.csv")

tables_brutes = {
    "convives": convives,
    "etablissements": etablissements,
    "menus": menus,
    "repas_servis": repas_servis
}

for nom, df in tables_brutes.items():
    print(f"\n--- {nom} ---")
    print("Dimensions :", df.shape)
    display(df.head())


---
## 2. Diagnostic (3 min)

Appliquez votre fonction `profile_dataframe` de la séance 2, puis **quantifiez** chaque
anomalie. Un diagnostic sans chiffre n'est pas un diagnostic.

In [ ]:
# Utiliser la fonction du cours si le module existe dans src/.
# Sinon, appliquer un profilage simple équivalent pour ne pas bloquer le TP.
import sys
sys.path.insert(0, str(ROOT / "src"))

try:
    from exploration import profile_dataframe
except ImportError:
    def profile_dataframe(df, name="dataframe"):
        n, p = df.shape
        print(f"Table : {name}")
        print(f"Dimensions : {n} lignes × {p} colonnes")
        print("\nTypes :")
        display(df.dtypes.rename("type").to_frame())
        print("\nValeurs manquantes par colonne :")
        display(df.isna().sum().rename("manquants").to_frame())
        print(f"\nLignes entièrement dupliquées : {df.duplicated().sum()}")
        print("\nStatistiques descriptives :")
        display(df.describe(include="all").T)


In [ ]:
# Diagnostic initial : on mesure avant de corriger
tables = {
    "convives": convives,
    "etablissements": etablissements,
    "menus": menus,
    "repas_servis": repas_servis
}

for nom, df in tables.items():
    print(f"\n{'=' * 60}\nDIAGNOSTIC : {nom}\n{'=' * 60}")
    profile_dataframe(df, name=nom)

# Indicateurs chiffrés à reporter dans le tableau des anomalies
for nom, df in tables.items():
    total = df.size
    manquants = int(df.isna().sum().sum())
    print(
        f"{nom}: {df.shape[0]} lignes, {df.shape[1]} colonnes, "
        f"{manquants} cellules manquantes, "
        f"{df.duplicated().sum()} lignes dupliquées"
    )


### Tableau des anomalies observées

| Anomalie | Colonne | Ampleur | Gravité | Traitement retenu |
|---|---|---:|---|---|
| Âges manquants | `convives.age` | 111 / 2 270 (4,9 %) | Moyenne | Valeurs manquantes conservées dans la source ; âge imputé par médiane après mise à l’écart des âges impossibles dans la table de travail |
| Âges impossibles | `convives.age` | 41 valeurs hors intervalle 0–110 | Élevée | Remplacer par manquant puis imputer par médiane valide (40 ans) ; conserver la source brute |
| Coûts de menu manquants | `menus.cout_reference_eur` | 5 / 180 (2,8 %) | Faible à moyenne | Imputation par médiane de la colonne, avec traçabilité dans la source brute |
| Dates manquantes | `repas_servis.date_evenement` | 377 / 18 683 (2,0 %) | Élevée pour les analyses temporelles | Ne pas inventer de date ; conserver manquant et exclure uniquement des agrégations par date |
| Grammage manquant | `repas_servis.grammage_g` | 2 050 / 18 683 (11,0 %) | Moyenne | Conserver manquant plutôt que d’inventer une quantité individuelle |
| Doublons exacts | `repas_servis` | 220 lignes (1,2 %) | Élevée | Supprimer les copies exactes dans la table de travail |
| Coût au format texte mixte | `repas_servis.cout_revient_eur` | 1 305 valeurs avec virgule ou suffixe EUR | Moyenne | Convertir en numérique après normalisation du séparateur décimal et du suffixe |
| Casse/espaces incohérents | `repas_servis.regime`, `repas_servis.statut`, `convives.ville` | Plusieurs variantes pour les mêmes libellés | Moyenne | Supprimer les espaces périphériques et uniformiser la casse |
| Portions atypiques | `repas_servis.portions_servies` | 83 valeurs hors intervalle observé 1–4 | À vérifier métier | Ajouter un indicateur de contrôle ; ne pas supprimer sans règle métier confirmée |

Les contrôles de clés étrangères sur les trois identifiants de `repas_servis` ne détectent aucune référence absente dans les référentiels fournis. Les nombres ci-dessus sont calculés sur les fichiers bruts.


---
## 3. Traitements (5 min — la partie la plus longue de la soutenance)

Pour chaque décision, trois éléments sont attendus : **ce que vous avez fait**, **pourquoi**,
et **quelle alternative vous avez écartée**. Le troisième point est celui qui fait la
différence à l'oral.

In [ ]:
# Pipeline de nettoyage et de transformation
# Les fichiers bruts ne sont jamais écrasés.

convives_clean = convives.copy()
etablissements_clean = etablissements.copy()
menus_clean = menus.copy()
repas_clean = repas_servis.copy()

# 1) Normaliser les textes de ville, régime et statut
for col in ["ville"]:
    convives_clean[col] = convives_clean[col].astype("string").str.strip().str.title()

for col in ["ville", "nom_site", "region"]:
    etablissements_clean[col] = etablissements_clean[col].astype("string").str.strip()

for col in ["libelle", "famille", "sous_famille"]:
    menus_clean[col] = menus_clean[col].astype("string").str.strip()

for col in ["regime", "statut"]:
    repas_clean[col] = repas_clean[col].astype("string").str.strip().str.lower()

# 2) Dates : conversion en datetime ; les dates absentes restent absentes
convives_clean["date_creation"] = pd.to_datetime(
    convives_clean["date_creation"], errors="coerce"
)
repas_clean["date_evenement"] = pd.to_datetime(
    repas_clean["date_evenement"], errors="coerce"
)

# 3) Âges impossibles : les traiter comme manquants, puis imputer par la médiane valide
convives_clean["age"] = pd.to_numeric(convives_clean["age"], errors="coerce")
ages_valides = convives_clean["age"].where(convives_clean["age"].between(0, 110))
mediane_age = ages_valides.median()
convives_clean["age"] = ages_valides.fillna(mediane_age)

# 4) Coûts : convertir les coûts de repas en numérique malgré les formats mixtes
repas_clean["cout_revient_eur"] = (
    repas_clean["cout_revient_eur"].astype("string")
    .str.replace(" EUR", "", regex=False)
    .str.replace(",", ".", regex=False)
)
repas_clean["cout_revient_eur"] = pd.to_numeric(
    repas_clean["cout_revient_eur"], errors="coerce"
)

# 5) Coût de référence manquant : imputation par médiane
menus_clean["cout_reference_eur"] = pd.to_numeric(
    menus_clean["cout_reference_eur"], errors="coerce"
)
mediane_cout_menu = menus_clean["cout_reference_eur"].median()
menus_clean["cout_reference_eur"] = menus_clean["cout_reference_eur"].fillna(
    mediane_cout_menu
)

# 6) Doublons exacts : retirer uniquement les copies identiques
nb_doublons_avant = int(repas_clean.duplicated().sum())
repas_clean = repas_clean.drop_duplicates().copy()

# 7) Contrôle des portions : on signale les valeurs atypiques sans les modifier
repas_clean["portions_hors_plage_observee"] = ~repas_clean["portions_servies"].between(1, 4)

# 8) Conserver les indicateurs de valeurs manquantes utiles à l'interprétation
repas_clean["date_manquante"] = repas_clean["date_evenement"].isna()
repas_clean["grammage_manquant"] = repas_clean["grammage_g"].isna()

# 9) Construire un jeu enrichi par jointures (relations vérifiées sur les identifiants)
jeu_final = (
    repas_clean
    .merge(menus_clean, on="id_menu", how="left", validate="many_to_one")
    .merge(etablissements_clean, on="id_etablissement", how="left", validate="many_to_one")
    .merge(
        convives_clean[["id_convive", "segment", "ville", "age"]],
        on="id_convive", how="left", validate="many_to_one",
        suffixes=("", "_convive")
    )
)

print("Médiane d'âge utilisée :", mediane_age)
print("Médiane du coût de référence utilisée :", round(float(mediane_cout_menu), 2))
print("Doublons exacts retirés :", nb_doublons_avant)
print("Dimensions avant nettoyage :", repas_servis.shape)
print("Dimensions après dédoublonnage :", repas_clean.shape)
print("Dimensions du jeu final enrichi :", jeu_final.shape)

display(jeu_final.head())


### Journal des décisions

| Décision | Justification | Alternative écartée | Motif du rejet |
|---|---|---|---|
| Normaliser les libellés (`strip` + `lower` ou `title`) | Éviter de compter séparément `STANDARD`, `standard` et ` standard ` | Garder les libellés bruts tels quels | Cela créerait de fausses catégories dans les graphiques |
| Imputer les âges impossibles/manquants par la médiane valide (40 ans) | La médiane résiste mieux aux valeurs extrêmes que la moyenne | Utiliser la moyenne brute | Les valeurs -8 et 999 faussent la moyenne |
| Convertir le coût en numérique | Le coût contient des virgules et le suffixe `EUR` | Garder le coût en texte | Impossible de calculer correctement moyenne, somme ou coût par statut |
| Supprimer les 220 doublons exacts | Les lignes sont identiques sur toutes les colonnes | Supprimer toutes les lignes partageant seulement un identifiant | Une même clé pourrait correspondre à une anomalie à examiner ; ici on retire seulement les copies exactes |
| Ne pas imputer la date et le grammage des repas | Aucune valeur fiable ne permet de les reconstituer individuellement | Remplacer par une date ou un grammage arbitraire | Cela inventerait une information métier |
| Signaler les portions hors plage 1–4 | 83 valeurs sont atypiques par rapport à la majorité des observations | Les remplacer automatiquement par 1–4 | Une règle métier doit confirmer si ces valeurs sont réellement impossibles |

Les imputations sont appliquées uniquement aux tables de travail. Les CSV d’origine restent inchangés.


---
## 4. Composant PySpark (2 min)

Quelle étape avez-vous portée sur Spark, et **pourquoi celle-là** ? Chiffrez : volume
traité, durée, mémoire. Rappel de la séance 7 : « parce que c'est plus rapide » est une
réponse fausse sur une machine unique.

In [ ]:
# Sécurité : permet aussi d'exécuter cette cellule seule après un redémarrage du noyau.
if "PROCESSED_DIR" not in globals():
    from pathlib import Path
    MAC_ROOT = Path("/Users/benjaminrenard/Downloads/Master Data : IA Python Pandas & Numpy/Master-Data-IA")
    ROOT = MAC_ROOT if MAC_ROOT.exists() else (Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd())
    RAW_CANDIDATES = [ROOT / "data" / "RAW", ROOT / "data" / "raw", ROOT / "raw"]
    RAW_DIR = next((p for p in RAW_CANDIDATES if p.exists()), RAW_CANDIDATES[0])
    PROCESSED_DIR = ROOT / "data" / "processed"
    PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

# Étape PySpark et mesures associées
# Cette étape démontre une agrégation distribuée reproductible.
# Le jeu est modeste : Spark n'est pas supposé être plus rapide que Pandas sur ce volume.

from time import perf_counter
import os

# Le jeu final a été produit à la section précédente.
spark_input = PROCESSED_DIR / "repas_servis_nettoye.csv"
repas_clean.to_csv(spark_input, index=False)

memoire_octets = int(repas_clean.memory_usage(deep=True).sum())
print(f"Volume traité : {len(repas_clean):,} lignes")
print(f"Mémoire estimée du DataFrame Pandas : {memoire_octets / 1024**2:.2f} Mio")
print(f"Taille du CSV d'entrée Spark : {spark_input.stat().st_size / 1024**2:.2f} Mio")

try:
    from pyspark.sql import SparkSession
    from pyspark.sql import functions as F

    spark = (
        SparkSession.builder
        .appName("ProjetCantine")
        .master("local[*]")
        .getOrCreate()
    )

    debut = perf_counter()
    repas_spark = (
        spark.read
        .option("header", True)
        .option("inferSchema", True)
        .csv(str(spark_input))
    )
    nb_lignes_spark = repas_spark.count()
    synthese_statuts = (
        repas_spark.groupBy("statut")
        .agg(F.count("*").alias("nombre_repas"))
        .orderBy(F.desc("nombre_repas"))
    )
    synthese_statuts.show(truncate=False)
    duree_spark = perf_counter() - debut

    print(f"Nombre de lignes lu par Spark : {nb_lignes_spark:,}")
    print(f"Durée mesurée (lecture + count + agrégation) : {duree_spark:.2f} s")
    print("Interprétation : mesure indicative sur cette machine, pas une preuve que Spark est plus rapide.")
except ImportError:
    print("PySpark n'est pas installé dans cet environnement.")
    print("Pour exécuter cette section, lancer le notebook dans l'environnement .venv-pyspark.")


---
## 5. Résultat (3 min)

Le jeu final, sa structure, les indicateurs, la planche de visualisation, et les limites
que vous assumez. **Énoncer soi-même les limites de son travail est valorisé** : cela
montre que vous savez où il ne faut pas faire confiance à vos données.

In [ ]:
# Sécurité : permet aussi d'exécuter cette cellule seule après un redémarrage du noyau.
if "PROCESSED_DIR" not in globals():
    from pathlib import Path
    MAC_ROOT = Path("/Users/benjaminrenard/Downloads/Master Data : IA Python Pandas & Numpy/Master-Data-IA")
    ROOT = MAC_ROOT if MAC_ROOT.exists() else (Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd())
    RAW_CANDIDATES = [ROOT / "data" / "RAW", ROOT / "data" / "raw", ROOT / "raw"]
    RAW_DIR = next((p for p in RAW_CANDIDATES if p.exists()), RAW_CANDIDATES[0])
    PROCESSED_DIR = ROOT / "data" / "processed"
    PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

# Chargement du jeu final et contrôles de cohérence
# Le jeu enrichi est créé dans la section 3.
jeu_final = jeu_final.copy()

print("Dimensions du jeu final :", jeu_final.shape)
print("Nombre de doublons exacts :", jeu_final.duplicated().sum())
print("Références de menu manquantes :", jeu_final["libelle"].isna().sum())
print("Références d'établissement manquantes :", jeu_final["nom_site"].isna().sum())
print("Références de convive manquantes :", jeu_final["segment"].isna().sum())

# Indicateurs descriptifs pour la soutenance
print("\nRépartition des statuts :")
display(repas_clean["statut"].value_counts(dropna=False).rename_axis("statut").to_frame("nombre"))

print("\nNombre de repas par établissement :")
display(
    jeu_final.groupby("nom_site", dropna=False)
    .size().sort_values(ascending=False)
    .rename("nombre_repas").to_frame().head(10)
)

print("\nCoût moyen par statut (EUR) :")
display(
    repas_clean.groupby("statut")["cout_revient_eur"]
    .agg(["count", "mean", "median"])
    .round(2)
)

# Enregistrer le jeu final et les tables de travail
jeu_final.to_csv(PROCESSED_DIR / "jeu_final_cantine.csv", index=False)
repas_clean.to_csv(PROCESSED_DIR / "repas_servis_nettoye.csv", index=False)
convives_clean.to_csv(PROCESSED_DIR / "convives_nettoye.csv", index=False)
menus_clean.to_csv(PROCESSED_DIR / "menus_nettoye.csv", index=False)
print("\nFichiers écrits dans :", PROCESSED_DIR)


### Limites assumées

1. **377 dates et 2 050 grammages restent manquants** dans les repas : les analyses temporelles et nutritionnelles doivent signaler cette limite.
2. **Les portions hors plage observée (83 lignes) sont signalées, mais pas corrigées**, car la règle métier n’est pas fournie.
3. **Le jeu couvre deux années seulement et la provenance exacte n’est pas documentée** dans les fichiers fournis ; les résultats décrivent ce jeu, sans garantir qu’ils représentent toute l’activité réelle.


---
## 6. Contrôle avant dépôt

La cellule suivante vérifie que votre rendu est complet. Exécutez-la avant de déposer.

Le dépôt est attendu le **jour 5 à 13 h 30**, avant la reprise de l'après-midi : votre
composant PySpark se termine le matin même. Prévoyez trente minutes pour la mise en forme,
pas cinq.

In [ ]:
# Sécurité : permet aussi d'exécuter cette cellule seule après un redémarrage du noyau.
if "PROCESSED_DIR" not in globals():
    from pathlib import Path
    MAC_ROOT = Path("/Users/benjaminrenard/Downloads/Master Data : IA Python Pandas & Numpy/Master-Data-IA")
    ROOT = MAC_ROOT if MAC_ROOT.exists() else (Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd())
    RAW_CANDIDATES = [ROOT / "data" / "RAW", ROOT / "data" / "raw", ROOT / "raw"]
    RAW_DIR = next((p for p in RAW_CANDIDATES if p.exists()), RAW_CANDIDATES[0])
    PROCESSED_DIR = ROOT / "data" / "processed"
    PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

checks = {
    "README.md présent": (ROOT / "README.md").exists(),
    "dossier src/ présent": (ROOT / "src").exists(),
    "quatre fichiers Cantine présents": all(
        (RAW_DIR / f).exists()
        for f in ["convives.csv", "etablissements.csv", "menus.csv", "repas_servis.csv"]
    ),
    "jeu final produit": (PROCESSED_DIR / "jeu_final_cantine.csv").exists(),
}

width = max(len(label) for label in checks)
for label, is_ok in checks.items():
    print(f"{label:<{width}}  {'OK' if is_ok else 'MANQUANT'}")

if all(checks.values()):
    print("\nContrôles disponibles : OK.")
else:
    print("\nCertains éléments restent à vérifier.")


## 7. Préparation aux questions

1. **Pourquoi imputer l’âge par la médiane ?** Les âges négatifs et 999 sont impossibles et faussent la moyenne. La médiane des âges valides est 40 ans. Les dates et grammages de repas ne sont pas imputés car aucune valeur fiable ne permet de les reconstituer.
2. **Comment justifier les lignes écartées ?** Seules les 220 lignes entièrement identiques sont dédoublonnées. Les portions atypiques ne sont pas supprimées. On conserve les données brutes et on compare les effectifs avant/après.
3. **Combien de colonnes l’encodage a-t-il produit ?** Aucun encodage one-hot n’est appliqué dans ce canevas, donc il n’y a pas de nombre final de colonnes encodées à annoncer. Si l’encodage est ajouté, relever `X_encoded.shape[1]`.
4. **À quel volume Pandas ne suffit plus ?** Il n’y a pas de seuil universel. Cela dépend de la mémoire et des opérations. Ici, le DataFrame `repas_servis` est de l’ordre de 18 000 lignes ; Spark sert surtout à démontrer une agrégation évolutive, pas à promettre un gain de vitesse.
5. **Que faire avec un nouveau fichier au même format ?** Vérifier les colonnes/types, exécuter le même pipeline, relancer les contrôles qualité et comparer les indicateurs.
6. **Quelle ligne traite le coût au format mixte ?** Le `str.replace(" EUR", "")`, puis `str.replace(",", ".")` et `pd.to_numeric(..., errors="coerce")` nettoient le texte avant conversion numérique.
7. **Quelle est la principale limite ?** Les dates manquantes (2,0 %) et les grammages manquants (11,0 %) empêchent certaines analyses complètes ; les portions atypiques demandent une validation métier.

> Les durées Spark et la mémoire affichées doivent être relevées après exécution dans votre environnement. Ne pas les inventer ou les recopier d’une autre machine.
